# Phase 2 — 유튜브 댓글 & 자막(STT) 수집

**수집 내용**
- `youtube_videos`: 영상 메타데이터 (제목, 채널, 조회수 등)
- `youtube_comments`: 영상 댓글
- `youtube_stt`: 유튜브 자동 자막 (라오어 품질 주의)

**파이프라인**
```
YouTube Data API v3 (영상 검색 → 댓글)
youtube-transcript-api (자막 추출)
         ↓
    DataFrame 정제
         ↓
  Supabase youtube_* 테이블
```

⚠️ **주의**: 라오어 자막은 자동 생성이 없거나 품질이 낮을 수 있음 → 수집 전 자막 가용 여부 확인

In [ ]:
# 필요 라이브러리 설치
# !pip install google-api-python-client youtube-transcript-api psycopg2-binary pandas python-dotenv

In [ ]:
import sys
sys.path.append('../')

import os
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from googleapiclient.discovery import build
from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound, TranscriptsDisabled
from src.db import insert_df, upsert_df

load_dotenv()
YOUTUBE_API_KEY = os.getenv('YOUTUBE_API_KEY')
youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)

## 1. 수집 대상 검색어 정의

In [ ]:
# 검색어 목록 (라오스·베트남·한국인 여행객 관점)
SEARCH_QUERIES = [
    {"query": "LOCA EV charging Laos",       "country": "LAO"},
    {"query": "LokaEV ລາວ",                  "country": "LAO"},  # 라오어
    {"query": "Green SM EV Laos",             "country": "LAO"},
    {"query": "V-Green EV Vietnam charging",  "country": "VNM"},
    {"query": "sạc điện V-Green VinFast",     "country": "VNM"},  # 베트남어
    {"query": "라오스 전기차 충전",             "country": "KOR"},  # 한국인 여행객
    {"query": "라오스 EV 충전소 리뷰",         "country": "KOR"},
]

MAX_RESULTS_PER_QUERY = 10  # 쿼리당 최대 영상 수
MAX_COMMENTS_PER_VIDEO = 100  # 영상당 최대 댓글 수

## 2. 유튜브 영상 검색 & 메타데이터 수집

In [ ]:
def search_videos(query: str, country: str, max_results: int = 10) -> pd.DataFrame:
    """YouTube Data API v3로 영상 검색 및 메타데이터 수집"""
    print(f"\n🔍 검색: '{query}'")
    
    try:
        # 영상 검색
        search_resp = youtube.search().list(
            q=query,
            part='id',
            type='video',
            maxResults=max_results,
            order='relevance',
        ).execute()
        
        video_ids = [item['id']['videoId'] for item in search_resp.get('items', [])]
        
        if not video_ids:
            print("  ⚠️ 검색 결과 없음")
            return pd.DataFrame()
        
        # 영상 상세 정보
        video_resp = youtube.videos().list(
            id=','.join(video_ids),
            part='snippet,statistics',
        ).execute()
        
        rows = []
        for item in video_resp.get('items', []):
            snippet = item['snippet']
            stats = item.get('statistics', {})
            rows.append({
                'video_id':     item['id'],
                'title':        snippet['title'],
                'channel_name': snippet['channelTitle'],
                'country':      country,
                'upload_date':  snippet['publishedAt'][:10],  # YYYY-MM-DD
                'view_count':   int(stats.get('viewCount', 0)),
                'like_count':   int(stats.get('likeCount', 0)),
                'description':  snippet.get('description', '')[:500],  # 500자 제한
            })
        
        df = pd.DataFrame(rows)
        print(f"  ✅ {len(df)}개 영상 수집")
        return df
    
    except Exception as e:
        print(f"  ❌ 오류: {e}")
        return pd.DataFrame()

In [ ]:
# 전체 검색어로 영상 수집
all_videos = []

for item in SEARCH_QUERIES:
    df = search_videos(item['query'], item['country'], MAX_RESULTS_PER_QUERY)
    if not df.empty:
        all_videos.append(df)

videos_df = pd.concat(all_videos, ignore_index=True).drop_duplicates('video_id') if all_videos else pd.DataFrame()
print(f"\n📊 총 수집 영상: {len(videos_df)}개 (중복 제거)")
print(videos_df.groupby('country').size())

In [ ]:
# DB 적재 (video_id 기준 중복 방지)
if not videos_df.empty:
    upsert_df(videos_df, 'youtube_videos', conflict_cols=['video_id'])
    print(videos_df[['video_id', 'title', 'country', 'view_count']].to_string())

## 3. 유튜브 댓글 수집

In [ ]:
def scrape_comments(video_id: str, max_comments: int = 100) -> pd.DataFrame:
    """YouTube Data API v3로 댓글 수집"""
    rows = []
    next_page_token = None
    
    try:
        while len(rows) < max_comments:
            resp = youtube.commentThreads().list(
                videoId=video_id,
                part='snippet',
                maxResults=min(100, max_comments - len(rows)),
                pageToken=next_page_token,
                textFormat='plainText',
                order='relevance',
            ).execute()
            
            for item in resp.get('items', []):
                c = item['snippet']['topLevelComment']['snippet']
                rows.append({
                    'video_id':     video_id,
                    'author':       c['authorDisplayName'],
                    'content':      c['textDisplay'],
                    'like_count':   c['likeCount'],
                    'comment_date': c['publishedAt'],
                })
            
            next_page_token = resp.get('nextPageToken')
            if not next_page_token:
                break
    
    except Exception as e:
        # 댓글 비활성화 영상 등
        if 'commentsDisabled' in str(e):
            print(f"  ⚠️ [{video_id}] 댓글 비활성화")
        else:
            print(f"  ❌ [{video_id}] 오류: {e}")
    
    return pd.DataFrame(rows) if rows else pd.DataFrame()

In [ ]:
# 수집된 영상 전체 댓글 수집
all_comments = []

for video_id in videos_df['video_id'].tolist():
    print(f"댓글 수집 중: {video_id}")
    df = scrape_comments(video_id, max_comments=MAX_COMMENTS_PER_VIDEO)
    if not df.empty:
        all_comments.append(df)
        print(f"  → {len(df)}개")

comments_df = pd.concat(all_comments, ignore_index=True) if all_comments else pd.DataFrame()
print(f"\n📊 총 댓글: {len(comments_df)}개")

In [ ]:
# DB 적재
if not comments_df.empty:
    insert_df(comments_df, 'youtube_comments')

## 4. 유튜브 자막(STT) 수집

⚠️ **라오어 자막 주의사항**
- 라오어 자동 자막이 없는 경우 영어/다른 언어 자막으로 대체
- 자막 없는 영상은 건너뜀

In [ ]:
def scrape_transcript(video_id: str) -> pd.DataFrame:
    """youtube-transcript-api로 자막 수집"""
    # 시도할 언어 우선순위
    lang_priority = ['lo', 'vi', 'th', 'en', 'ko']
    
    try:
        transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
        
        # 우선순위에 따라 자막 선택
        transcript = None
        for lang in lang_priority:
            try:
                transcript = transcript_list.find_transcript([lang])
                break
            except:
                continue
        
        # 어떤 자막도 없으면 자동 생성 자막 사용
        if transcript is None:
            try:
                transcript = transcript_list.find_generated_transcript(['en', 'lo', 'vi'])
            except:
                print(f"  ⚠️ [{video_id}] 사용 가능한 자막 없음")
                return pd.DataFrame()
        
        data = transcript.fetch()
        
        rows = [{
            'video_id':        video_id,
            'timestamp_start': item['start'],
            'timestamp_end':   item['start'] + item.get('duration', 0),
            'content':         item['text'].strip(),
        } for item in data if item['text'].strip()]
        
        return pd.DataFrame(rows)
    
    except TranscriptsDisabled:
        print(f"  ⚠️ [{video_id}] 자막 비활성화")
        return pd.DataFrame()
    except Exception as e:
        print(f"  ❌ [{video_id}] 오류: {e}")
        return pd.DataFrame()

In [ ]:
# 전체 자막 수집
all_stt = []
stt_success = 0
stt_fail = 0

for video_id in videos_df['video_id'].tolist():
    df = scrape_transcript(video_id)
    if not df.empty:
        all_stt.append(df)
        stt_success += 1
        print(f"  ✅ [{video_id}] {len(df)}개 자막 세그먼트")
    else:
        stt_fail += 1

print(f"\n📊 자막 수집 결과: 성공 {stt_success}개, 실패 {stt_fail}개")

stt_df = pd.concat(all_stt, ignore_index=True) if all_stt else pd.DataFrame()
print(f"총 자막 세그먼트: {len(stt_df)}개")

In [ ]:
# DB 적재
if not stt_df.empty:
    insert_df(stt_df, 'youtube_stt')

## 5. 수집 결과 확인

In [ ]:
from src.db import fetch_df

# 영상별 댓글·자막 수 확인
result = fetch_df("""
    SELECT 
        v.video_id,
        v.title,
        v.country,
        COUNT(DISTINCT c.id) AS comment_count,
        COUNT(DISTINCT s.id) AS stt_count
    FROM youtube_videos v
    LEFT JOIN youtube_comments c ON v.video_id = c.video_id
    LEFT JOIN youtube_stt s ON v.video_id = s.video_id
    GROUP BY v.video_id, v.title, v.country
    ORDER BY comment_count DESC
""")

print("📊 DB 저장 현황")
print(result.to_string())